# Tutorial: Data Cleaning & Wrangling

> **Type:** Tutorial. Follow along step by step.  
> **Time:** ~2 hours  
> **Prerequisites:** Module 06 (pandas basics)  
> **What you'll build:** A complete data cleaning pipeline that takes a deliberately messy employee dataset from raw (errors everywhere) to analysis-ready, with before/after comparisons at every step.

---

## What We're Building

We'll generate a messy synthetic employee dataset that contains every common data quality problem: missing values, outliers, duplicates, wrong types, and inconsistent categories. Then we'll clean it step by step, checking each step. By the end, you'll have a reusable cleaning pattern you can apply to any real dataset.

## Step 0: Setup

In [ ]:
# !pip install pandas numpy matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.impute import KNNImputer

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 4)
np.random.seed(42)

print('Setup complete.')

## Step 1: Generate a Messy Dataset

We're building the mess ourselves, so we know *exactly* what problems exist. This is a common technique when learning cleaning: you damage the dataset on purpose, then prove you can fix it.

In [ ]:
n = 500

# Generate base data
departments = np.random.choice(['Engineering', 'Sales', 'Marketing', 'HR', 'Finance'], n)
genders_clean = np.random.choice(['male', 'female'], n)
office_cities = np.random.choice(
    ['New York', 'London', 'São Paulo', 'Zürich', 'Montréal'], n
)

# ---- Introduce deliberate problems ----

# 1. Inconsistent gender categories
gender_variants = {
    'male':   ['male', 'Male', 'M', 'man', 'MALE'],
    'female': ['female', 'Female', 'F', 'woman', 'FEMALE'],
}
genders_messy = [
    np.random.choice(gender_variants[g]) for g in genders_clean
]

# 2. Ages with some missing and one clearly wrong value
ages = np.random.randint(22, 65, n).astype(float)
missing_age_idx = np.random.choice(n, 40, replace=False)
ages[missing_age_idx] = np.nan
ages[0] = 312  # typo outlier

# 3. Salary with outliers and some missing
salaries = np.random.normal(70_000, 15_000, n)
salaries = np.clip(salaries, 30_000, 200_000)
salaries[np.random.choice(n, 5, replace=False)] = 999_999  # data entry errors
missing_sal_idx = np.random.choice(n, 25, replace=False)
salaries[missing_sal_idx] = np.nan

# 4. Years of experience stored as string
experience = np.random.randint(0, 30, n)
experience_str = [str(e) for e in experience]  # wrong type

# 5. Hire date with some invalid values -- schema violations: each one is
# a syntactically fine string, but not a valid date, which is the actual
# constraint this column has to satisfy
hire_years = np.random.randint(2000, 2024, n)
hire_dates = [f'2023-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}' 
              if yr == 2023 else f'{yr}-{np.random.randint(1,13):02d}-01'
              for yr in hire_years]
hire_dates[10] = 'not a date'  # invalid
hire_dates[20] = ''            # empty string

# 6. Office city with encoding errors: a handful of non-ASCII city names
# get mangled the way this actually happens in practice -- UTF-8 bytes
# read back as if they were Latin-1
office_cities_messy = list(office_cities)
mojibake_idx = np.random.choice(n, 8, replace=False)
for i in mojibake_idx:
    office_cities_messy[i] = office_cities_messy[i].encode('utf-8').decode('latin1')

# Build DataFrame
df_raw = pd.DataFrame({
    'employee_id':  range(1, n + 1),
    'department':   departments,
    'gender':       genders_messy,
    'age':          ages,
    'salary':       salaries,
    'years_exp':    experience_str,   # wrong type
    'hire_date':    hire_dates,
    'office_city':  office_cities_messy,
})

# 7. Add duplicate rows (5 exact duplicates)
dupe_rows = df_raw.iloc[:5].copy()
df_raw = pd.concat([df_raw, dupe_rows], ignore_index=True)
df_raw = df_raw.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head(10)


## Step 2: Initial Inspection — The Diagnostic

Never start cleaning before you fully understand what you're dealing with.

In [ ]:
print('=== SHAPE ===')
print(f'{df_raw.shape[0]} rows × {df_raw.shape[1]} columns')

print('\n=== DTYPES ===')
print(df_raw.dtypes)

print('\n=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
print(pd.DataFrame({'count': missing, 'pct': missing_pct})[missing > 0])

print('\n=== DUPLICATES ===')
print(f'Exact duplicate rows: {df_raw.duplicated().sum()}')

print('\n=== CATEGORY DISTRIBUTIONS ===')
print('gender:', df_raw['gender'].value_counts().to_dict())
print('office_city:', df_raw['office_city'].value_counts().to_dict())

In [ ]:
df_raw.describe()

> **What just happened?** The diagnostic reveals all seven problem types:
> - wrong type (`years_exp`)
> - inconsistent categories (`gender`)
> - missing values (`age`, `salary`)
> - outliers (`age` max = 312, `salary` max = 999,999)
> - duplicates (5 rows)
> - schema violations (`hire_date` values that are valid strings but not valid dates)
> - encoding errors (`office_city` values mangled by a UTF-8/Latin-1 mismatch, so one city shows up as two different-looking categories)
>
> We'll fix them in order.

## Step 3: Remove Duplicates

In [ ]:
df = df_raw.copy()  # always work on a copy; keep the raw data intact

before = len(df)
df = df.drop_duplicates()
df = df.reset_index(drop=True)
after = len(df)

print(f'Rows before: {before}')
print(f'Rows after:  {after}')
print(f'Removed:     {before - after} duplicate rows')

## Step 4: Fix Column Types

In [ ]:
print('Types before fixing:')
print(df[['years_exp', 'hire_date']].dtypes)

# years_exp: string → integer
df['years_exp'] = pd.to_numeric(df['years_exp'], errors='coerce')  # errors='coerce' turns unconvertible values to NaN

# hire_date: string → datetime (bad values become NaT)
df['hire_date'] = pd.to_datetime(df['hire_date'], errors='coerce')

print('\nTypes after fixing:')
print(df[['years_exp', 'hire_date']].dtypes)

# Check how many hire_dates became NaT
print(f'\nInvalid hire_dates converted to NaT: {df["hire_date"].isnull().sum()}')

## Step 5: Normalise Inconsistent Categories

In [ ]:
print('Gender values before normalisation:')
print(df['gender'].value_counts())

# Step 1: strip whitespace and lowercase
df['gender'] = df['gender'].str.strip().str.lower()

# Step 2: map all variants to canonical forms
gender_map = {
    'male': 'male', 'm': 'male', 'man': 'male',
    'female': 'female', 'f': 'female', 'woman': 'female',
}
df['gender'] = df['gender'].map(gender_map)

print('\nGender values after normalisation:')
print(df['gender'].value_counts())

## Step 5b: Repair Encoding Errors

A city name that should be one category shows up as two. Some rows were corrupted by a classic encoding mismatch: UTF-8 bytes decoded as if they were Latin-1. The fix repeats how the corruption happened, in reverse.

In [ ]:
print('office_city values before repair:')
print(df['office_city'].value_counts())

def fix_mojibake(s):
    try:
        return s.encode('latin1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return s

df['office_city'] = df['office_city'].apply(fix_mojibake)

print('\noffice_city values after repair:')
print(df['office_city'].value_counts())

## Step 6: Handle Outliers

We'll use both the IQR method and a domain-knowledge check.

In [ ]:
def detect_outliers_iqr(series, factor=1.5):
    """Return boolean mask: True where value is an outlier."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    return (series < lower) | (series > upper)

# Age: flag + domain-knowledge cap (no employee can be >90 or <16)
age_outliers = detect_outliers_iqr(df['age'].dropna())
print(f'Age outliers detected by IQR: {age_outliers.sum()}')
print(f'Age max before: {df["age"].max()}')

# Cap at sensible domain limits
df['age'] = df['age'].clip(lower=16, upper=90)
print(f'Age max after:  {df["age"].max()}')

In [ ]:
# Salary: winsorise at 1st and 99th percentile
salary_p01 = df['salary'].quantile(0.01)
salary_p99 = df['salary'].quantile(0.99)

print(f'Salary before winsorising — min: {df["salary"].min():.0f}, max: {df["salary"].max():.0f}')
df['salary'] = df['salary'].clip(lower=salary_p01, upper=salary_p99)
print(f'Salary after winsorising  — min: {df["salary"].min():.0f}, max: {df["salary"].max():.0f}')

# Visualise before vs after on the outlier removal
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_raw['salary'].dropna().plot(kind='hist', bins=30, ax=axes[0],
    color='#E15759', edgecolor='white', title='Salary — BEFORE')
df['salary'].dropna().plot(kind='hist', bins=30, ax=axes[1],
    color='#59A14F', edgecolor='white', title='Salary — AFTER Winsorising')
for ax in axes:
    ax.set_xlabel('Salary ($)')
plt.tight_layout()
plt.savefig('salary_outliers.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Two salary histograms. Before winsorising, one value of 999,999 stretches the x-axis to a million, so almost every salary sits in two bars at the far left. After clipping to the 1st and 99th percentiles, salaries run from about 33,000 to 113,000 in a rough bell shape centred near 70,000.

## Step 7: Handle Missing Values

In [ ]:
print('Missing values before imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Strategy A: Median imputation for age (robust to skew)
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)
print(f'Age median used for imputation: {age_median:.0f}')

# Strategy B: KNN imputation for salary (uses other features to estimate)
# We'll use age and years_exp to estimate missing salaries
impute_cols = ['age', 'salary', 'years_exp']
imputer = KNNImputer(n_neighbors=5)

df_to_impute = df[impute_cols].copy()
imputed = imputer.fit_transform(df_to_impute)
df['salary'] = imputed[:, 1]  # column index 1 = salary

# Strategy C: Drop rows with invalid hire_date (only 2 rows)
df = df.dropna(subset=['hire_date'])

print('\nMissing values after imputation:')
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.any() else 'None — all columns complete!')

## Step 8: Validate the Cleaned Dataset

In [ ]:
print('=== CLEANED DATASET VALIDATION ===')
print(f'Rows: {len(df)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')

print('\nAge range:', df['age'].min(), '–', df['age'].max())
print('Salary range: ${:,.0f} – ${:,.0f}'.format(df['salary'].min(), df['salary'].max()))
print('\nGender categories:', df['gender'].unique())
print('\nOffice city categories:', sorted(df['office_city'].unique()))
print('\nDtypes:')
print(df.dtypes)

In [ ]:
# Visual validation: compare distributions before and after
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Age distribution
df_raw['age'].dropna().plot(kind='hist', bins=25, ax=axes[0,0],
    color='#E15759', edgecolor='white', alpha=0.7)
axes[0,0].set_title('Age — BEFORE')
df['age'].plot(kind='hist', bins=25, ax=axes[0,1],
    color='#59A14F', edgecolor='white', alpha=0.7)
axes[0,1].set_title('Age — AFTER')

# Salary distribution
df_raw['salary'].dropna().plot(kind='hist', bins=25, ax=axes[1,0],
    color='#E15759', edgecolor='white', alpha=0.7)
axes[1,0].set_title('Salary — BEFORE')
df['salary'].plot(kind='hist', bins=25, ax=axes[1,1],
    color='#59A14F', edgecolor='white', alpha=0.7)
axes[1,1].set_title('Salary — AFTER')

for ax in axes.flat:
    ax.set_xlabel('Value')

plt.tight_layout()
plt.savefig('cleaning_before_after.png', dpi=120, bbox_inches='tight')
plt.show()

print('Cleaned dataset saved to df — ready for modelling.')

**Chart description.** Four histograms of age and salary, before and after cleaning. Before, an age of 312 and a salary of 999,999 stretch the axes and squash the real values into a few bars. After, ages run from 22 to 65, with a spike at 42 from the missing ages filled with the median and one value at 90 where the 312 was capped. Salaries after cleaning form a bell shape from about 33,000 to 113,000, centred near 70,000.

## Step 9: Wrap the Remaining Cleaning in a Pipeline (Production-Ready)

Some of the steps above are safe to run before a train/test split, because they don't use any statistic computed across rows: deduplication, type-casting, category normalisation, the encoding repair, and the age cap at fixed domain limits (16–90). Others aren't. The salary winsorising in Step 6 clipped at the *whole dataset's* 1st and 99th percentiles, and imputation fills gaps with a mean, median, or KNN estimate. Fitting either on the full dataset leaks test-set information into training, the same kind of leakage Module 02 warns about. We kept them simple above to show the mechanics. From here on, split first, then wrap every step that learns from the data in a `ColumnTransformer`, so leakage is impossible by construction, not just by discipline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# Redo Steps 3-6 (safe to do before splitting) starting from df_raw,
# stopping just short of the imputation Step 7 already did in place
df_pre_impute = df_raw.drop_duplicates().reset_index(drop=True).copy()
df_pre_impute['years_exp'] = pd.to_numeric(df_pre_impute['years_exp'], errors='coerce')
df_pre_impute['hire_date'] = pd.to_datetime(df_pre_impute['hire_date'], errors='coerce')
df_pre_impute['gender'] = df_pre_impute['gender'].str.strip().str.lower().map(gender_map)
df_pre_impute['age'] = df_pre_impute['age'].clip(lower=16, upper=90)
df_pre_impute['salary'] = df_pre_impute['salary'].clip(
    lower=df_pre_impute['salary'].quantile(0.01),
    upper=df_pre_impute['salary'].quantile(0.99),
)

train_df, test_df = train_test_split(df_pre_impute, test_size=0.2, random_state=42)

numeric_cols = ['age', 'salary', 'years_exp']
categorical_cols = ['gender']

numeric_pipeline = Pipeline([
    ('impute', KNNImputer(n_neighbors=5)),
])
categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

# Fit only on the training split
train_clean = preprocessor.fit_transform(train_df[numeric_cols + categorical_cols])
# Test data only ever gets transformed, never fit
test_clean = preprocessor.transform(test_df[numeric_cols + categorical_cols])

print(f'Train shape after Pipeline: {train_clean.shape}')
print(f'Test shape after Pipeline:  {test_clean.shape}')
print(f'Missing values remaining in train (numeric cols): '
      f'{np.isnan(train_clean[:, :len(numeric_cols)]).sum()}')
print(f'Missing values remaining in test (numeric cols):  '
      f'{np.isnan(test_clean[:, :len(numeric_cols)]).sum()}')

## What You Built

A complete cleaning pipeline. It took a deliberately messy 505-row dataset through deduplication, type fixing, category normalisation, an encoding repair, outlier capping, and two different imputation strategies, with before/after comparisons at each step. It then wrapped the leakage-sensitive steps in a `ColumnTransformer` fit only on a training split, the production-ready version of the same pattern. You'll repeat this pattern on every real dataset.

## What's Next

- Read `explanation.md` for the theory behind missing-data mechanisms (MCAR/MAR/MNAR) and when each imputation strategy fits.
- `howto.md` has one-line recipes for each cleaning operation.
- `reference.md` is a cheat sheet for all the pandas cleaning commands.
- **Extension challenge:** add a schema validation step. Use assertions (`assert (df['age'] > 0).all()`) to check every business rule holds after cleaning.

---

**Next module:** `08_sql_for_data_science`, because you often need to clean and aggregate data before it even reaches Python.